# Découverte de Pattern Functional Dependencies (PFDs) assistée par LLM

## Contexte

Une **Pattern Functional Dependency (PFD)** est une généralisation des dépendances 
fonctionnelles classiques : au lieu de travailler sur les valeurs brutes, elle s'applique 
sur des *patterns* extraits de ces valeurs.

**Exemple :** `prefix(ZIP, 3) → CITY`  
Les 3 premiers chiffres d'un code postal déterminent la ville.

L'enjeu de la découverte de PFDs est double :
- **Profiling de données** : comprendre la structure implicite d'un dataset
- **Qualité des données** : détecter des incohérences ou anomalies

L'approche classique (force brute) teste toutes les combinaisons possibles 
(colonnes × transformations), ce qui devient coûteux sur des datasets larges. 
Ce projet explore l'apport des LLMs pour guider et accélérer cette découverte.

## Transformations disponibles

Les valeurs de chaque colonne peuvent être transformées avant d'évaluer une dépendance :

| Transformation | Description | Exemple |
|---|---|---|
| `raw` | Valeur brute | `"New York"` |
| `prefix_n` | n premiers caractères | `prefix_3("90210")` → `"902"` |
| `suffix_n` | n derniers caractères | `suffix_2("gmail.com")` → `"om"` |
| `first_token` | Premier mot | `"John Smith"` → `"John"` |
| `last_token` | Dernier mot | `"John Smith"` → `"Smith"` |
| `domain` | Domaine d'un email | `"u@gmail.com"` → `"gmail.com"` |
| `uppercase` / `lowercase` | Normalisation de casse | `"NY"` → `"ny"` |

## Trois workflows comparés

### Workflow classique
Force brute : toutes les paires (colonne × transformation) sont testées.  
Exhaustif mais lent — plusieurs centaines de candidats par dataset.

### Workflow agentique v1
Le LLM analyse les colonnes et suggère les transformations pertinentes *par colonne* 
avant la découverte. L'algorithme n'explore que ce sous-ensemble.  
→ Réduction du nombre de candidats testés (~50-70%)

### Workflow agentique v2 — Guided Search
Le LLM propose directement des **paires complètes** `(colonne source, transformation, colonne cible)`.  
L'algorithme valide uniquement ces paires — aucune exploration.  
→ 8 à 15 candidats testés au lieu de plusieurs centaines

## Providers LLM et modèles

| Provider | Modèle | Taille |
|---|---|---|
| Groq | `llama-3.3-70b-versatile` | 70B |
| HuggingFace | `Qwen2.5-7B-Instruct` | 7B |
| Mistral | `mistral-large-latest` | ~123B |


## Datasets utilisés

| Dataset | Lignes | Colonnes | Domaine |
|---|---|---|---|
| `t1.csv` | 9 101 | 17 | Employés municipaux |
| `t2.csv` | 3 501 | 9 | Entreprises (adresses, contacts) |
| `t3.csv` | 1 076 | 10 | Licences commerciales |
| `US_Phone_Code.csv` | 51 | 3 | Codes téléphoniques US |

## Métriques d'évaluation

- **nb_pfds** : nombre de PFDs découvertes
- **candidates_tested** : nombre de paires évaluées algorithmiquement
- **validation_rate** : `nb_pfds / candidates_tested` — proportion de paires proposées qui tiennent statistiquement (pertinent surtout pour v2)
- **mean_confidence** : confiance moyenne des PFDs découvertes
- **mean_support** : support moyen (nombre de lignes couvertes)
- **execution_time** : temps d'exécution total en secondes

Une PFD est retenue si `confidence ≥ 0.85` et `support ≥ 10`.

Le code suivant permet d'afficher toutes les pfd trouvées par les différents algorithmes et modèles pour le dataset indiqué dans la variable DATASET.

In [ ]:
import json
import os
import pandas as pd
from IPython.display import display

from pathlib import Path

DATASET     = "t1.csv"
RESULTS_DIR = Path().resolve().parent / "results"
# ───────────────────────────────────────────────────────────────

def load_pfds(dataset, workflow, results_dir):
    results = {}
    prefix = f"{dataset}_{workflow}_"

    for fname in sorted(os.listdir(results_dir)):
        if not fname.startswith(prefix) or not fname.endswith(".json"):
            continue
        provider = fname[len(prefix):-len(".json")]
        with open(os.path.join(results_dir, fname), encoding="utf-8") as f:
            data = json.load(f)

        runs = data.get("runs", [])
        n_runs = len(runs)

        # Compter dans combien de runs chaque PFD apparaît
        counts = {}
        values = {}
        for run in runs:
            for pfd in run.get("discovered_pfds", []):
                key = (pfd["lhs"], pfd["rhs"])
                counts[key] = counts.get(key, 0) + 1
                values[key] = pfd  # on garde les métriques du dernier run

        pfds = []
        for key, pfd in values.items():
            pfds.append({**pfd, "found_in_runs": f"{counts[key]}/{n_runs}"})

        results[provider] = {
            "pfds":    pfds,
            "nb_runs": n_runs,
            "params":  data.get("params", {}),
        }
    return results


def afficher_pfds(pfds, titre):
    print(f"\n{'='*60}")
    print(f"  {titre}")
    print(f"{'='*60}")
    if not pfds:
        print("  Aucune PFD trouvée.")
        return
    cols = ["lhs", "rhs", "support", "confidence", "found_in_runs"]
    df = (pd.DataFrame(pfds)[cols]
            .sort_values("confidence", ascending=False)
            .reset_index(drop=True))
    display(df)

In [ ]:
classique = load_pfds(DATASET, "classical", RESULTS_DIR)

if classique:
    data = classique.get("none", {})
    afficher_pfds(data.get("pfds", []), f"Classique — {DATASET}")
else:
    print(f"Aucun résultat classique trouvé pour {DATASET}")


  Classique — t1.csv


,lhs,rhs,support,confidence,found_in_runs
0,Full Name__raw,Gender,9101,1.0000,3/3
1,Full Name__raw,Underfilled Job Title,1092,1.0000,3/3
2,Full Name__raw,Assignment Category,9101,1.0000,3/3
3,Full Name__uppercase,Gender,9101,1.0000,3/3
4,Full Name__lowercase,Gender,9101,1.0000,3/3
...,...,...,...,...,...
178,Position Title__raw,Department,9101,0.8523,3/3
179,Position Title__lowercase,Department,9101,0.8523,3/3
180,Position Title__uppercase,Department Name,9101,0.8523,3/3
181,Position Title__uppercase,Department,9101,0.8523,3/3


In [ ]:
agent_v1 = load_pfds(DATASET, "agent_v1", RESULTS_DIR)

if agent_v1:
    for provider, data in agent_v1.items():
        titre = f"Agent v1 — {provider.upper()}  ({data['nb_runs']} run(s))"
        afficher_pfds(data["pfds"], titre)
else:
    print(f"Aucun résultat agent_v1 trouvé pour {DATASET}")


  Agent v1 — GROQ  (3 run(s))


,lhs,rhs,support,confidence,found_in_runs
0,Full Name__raw,Assignment Category,9101,1.0000,3/3
1,Full Name__raw,Gender,9101,1.0000,3/3
2,Full Name__raw,Underfilled Job Title,1092,1.0000,3/3
3,Department__raw,Department Name,9101,1.0000,3/3
4,Department__uppercase,Department Name,9101,1.0000,1/3
5,Full Name__raw,Date First Hired,9101,0.9993,3/3
6,Full Name__raw,Department Name,9101,0.9974,3/3
7,Full Name__raw,Department,9101,0.9974,3/3
8,Full Name__raw,Division,9101,0.9973,3/3
9,Full Name__raw,Position Title,9101,0.9971,3/3



  Agent v1 — HUGGINGFACE  (3 run(s))


,lhs,rhs,support,confidence,found_in_runs
0,Full Name__raw,Gender,9101,1.0000,3/3
1,Full Name__raw,Assignment Category,9101,1.0000,3/3
2,Full Name__raw,Underfilled Job Title,1092,1.0000,3/3
3,Department__raw,Department Name,9101,1.0000,3/3
4,Full Name__raw,Date First Hired,9101,0.9993,3/3
5,Full Name__raw,Department,9101,0.9974,3/3
6,Full Name__raw,Department Name,9101,0.9974,3/3
7,Full Name__raw,Division,9101,0.9973,3/3
8,Full Name__raw,Position Title,9101,0.9971,3/3
9,Underfilled Job Title__raw,Assignment Category,1092,0.9945,3/3



  Agent v1 — MISTRAL  (3 run(s))


,lhs,rhs,support,confidence,found_in_runs
0,Full Name__raw,Gender,9101,1.0000,3/3
1,Full Name__raw,Underfilled Job Title,1092,1.0000,3/3
2,Full Name__raw,Assignment Category,9101,1.0000,3/3
3,Department__uppercase,Department Name,9101,1.0000,3/3
4,Department__raw,Department Name,9101,1.0000,3/3
5,Full Name__raw,Date First Hired,9101,0.9993,3/3
6,Full Name__raw,Department,9101,0.9974,3/3
7,Full Name__raw,Department Name,9101,0.9974,3/3
8,Full Name__raw,Division,9101,0.9973,3/3
9,Full Name__raw,Position Title,9101,0.9971,3/3


In [ ]:
agent_v2 = load_pfds(DATASET, "agent_v2", RESULTS_DIR)

if agent_v2:
    for provider, data in agent_v2.items():
        titre = f"Agent v2 — {provider.upper()}  ({data['nb_runs']} run(s))"
        afficher_pfds(data["pfds"], titre)
else:
    print(f"Aucun résultat agent_v2 trouvé pour {DATASET}")


  Agent v2 — GROQ  (3 run(s))


,lhs,rhs,support,confidence,found_in_runs
0,Department__raw,Department Name,9101,1.0000,3/3
1,Department Name__raw,Department,9101,1.0000,2/3
2,Full Name__raw,Gender,9101,1.0000,1/3
3,Underfilled Job Title__raw,Position Title,1092,0.9808,1/3
4,Position Title__first_token,Assignment Category,9101,0.9580,1/3
5,Date First Hired__raw,Assignment Category,9101,0.9366,1/3
6,Underfilled Job Title__prefix_3,Position Title,1092,0.9121,1/3



  Agent v2 — HUGGINGFACE  (3 run(s))


,lhs,rhs,support,confidence,found_in_runs
0,Full Name__uppercase,Assignment Category,9101,1.0000,3/3
1,Full Name__lowercase,Underfilled Job Title,1092,1.0000,3/3
2,Department__uppercase,Department Name,9101,1.0000,1/3
3,Underfilled Job Title__first_token,Position Title,1092,0.9386,1/3
4,Date First Hired__first_token,Assignment Category,9101,0.9366,1/3
5,Division__first_token,Assignment Category,9101,0.9275,1/3



  Agent v2 — MISTRAL  (3 run(s))


,lhs,rhs,support,confidence,found_in_runs
0,Department__raw,Department Name,9101,1.0000,3/3
1,Department__uppercase,Department Name,9101,1.0000,1/3
2,Underfilled Job Title__prefix_3,Assignment Category,1092,0.9945,1/3
3,Position Title__prefix_3,Assignment Category,9101,0.9554,2/3
4,Division__first_token,Department Name,9101,0.9392,1/3
5,Division__first_token,Department,9101,0.9392,2/3
6,Underfilled Job Title__first_token,Position Title,1092,0.9386,1/3
7,Position Title__last_token,Assignment Category,9101,0.9244,1/3
8,Underfilled Job Title__prefix_3,Position Title,1092,0.9121,1/3
9,Division__prefix_3,Department,9101,0.9062,1/3
